# TinyDiffusion-ESP32: Complete ML Pipeline

This notebook implements a complete end-to-end pipeline for training a tiny conditional diffusion model on MNIST and deploying it to an ESP32 DevKit V1 (ESP32-WROOM-32) for offline image generation.

## Project Overview

- **Model**: Tiny class-conditional CNN (~3,277 parameters) for noise prediction
- **Dataset**: MNIST (28×28 grayscale digits 0-9)
- **Training**: DDPM with linear noise schedule (100 timesteps)
- **Sampling**: DDIM with configurable steps (8, 16, 32)
- **Quantization**: Full INT8 for TensorFlow Lite Micro
- **Target**: ESP32 DevKit V1 (520 KB SRAM, no PSRAM)
- **Display**: SSD1306 OLED via U8g2 library
- **Framework**: Arduino + TensorFlow Lite Micro + ESP-NN

## Architecture

```
Input: 28×28×12 (noisy image + 10 class maps + 1 timestep map)
├── Conv2D: 8 filters, 3×3, SAME, ReLU
├── Conv2D: 8 filters, 3×3, SAME, ReLU
├── Conv2D: 12 filters, 3×3, SAME, ReLU
├── Conv2D: 8 filters, 3×3, SAME, ReLU
└── Conv2D: 1 filter, 3×3, SAME, Linear
Output: 28×28×1 (predicted noise)
```

## Notebook Structure

1. **Setup & Configuration** - Dependencies, config, seeds
2. **Dataset** - MNIST download, preprocessing, visualization
3. **Diffusion** - Noise schedule, forward process
4. **Model** - Conditional CNN architecture
5. **Training** - DDPM training loop
6. **Sampling** - DDIM implementation
7. **Evaluation** - Generated samples, metrics, comparison
8. **Export** - TFLite float32 and INT8 conversion
9. **Verification** - Quantization accuracy, TFLite Micro compatibility
10. **Deployment** - C header generation for ESP32

⚠️ **Note**: Training cells are marked with warnings. Execute them only when ready for long-running computation.

In [ ]:
# Cell 1: Setup and Dependencies
import sys
import os
sys.path.insert(0, '../src')

# Check TensorFlow version
import tensorflow as tf
print(f"TensorFlow: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

# Install dependencies if needed (uncomment to run)
# !pip install -r ../requirements.txt

In [ ]:
# Cell 2: Configuration
import yaml
from src.utils import load_config, set_global_seed, ensure_dir

config = load_config("../configs/default.yaml")
print(yaml.dump(config, default_flow_style=False))

In [ ]:
# Cell 3: Set Random Seeds for Reproducibility
from src.utils import set_global_seed

seed = config['training']['seed']
set_global_seed(seed)
print(f"Global seed set to {seed}")

In [ ]:
# Cell 4: MNIST Dataset Download and Inspection
from src.dataset import MNISTDataset

dataset = MNISTDataset(config)
x_train, y_train, x_test, y_test = dataset.load()

# Preprocess
x_train_proc = dataset.preprocess(x_train)
x_test_proc = dataset.preprocess(x_test)

print(f"Train: {x_train_proc.shape}, range [{x_train_proc.min():.3f}, {x_train_proc.max():.3f}]")
print(f"Test: {x_test_proc.shape}, range [{x_test_proc.min():.3f}, {x_test_proc.max():.3f}]")

In [ ]:
# Cell 5: Dataset Visualization
dataset.visualize_samples(num_samples=16, save_path="../artifacts/mnist_samples.png")

In [ ]:
# Cell 6: Diffusion Schedule Implementation
from src.diffusion import NoiseSchedule, forward_diffusion, visualize_forward_diffusion

noise_schedule = NoiseSchedule(config)
print(f"Schedule: {noise_schedule}")
print(f"Betas: [{noise_schedule.betas.min():.6f}, {noise_schedule.betas.max():.6f}]")
print(f"Alpha_bar: [{noise_schedule.alphas_cumprod.min():.6f}, {noise_schedule.alphas_cumprod.max():.6f}]")

In [ ]:
# Cell 7: Forward Diffusion Visualization
# Show forward diffusion process on a sample image
sample_img = x_train_proc[0:1]  # [1, 28, 28, 1]
visualize_forward_diffusion(sample_img[0], noise_schedule, num_steps=10,
                           save_path="../artifacts/forward_diffusion.png")

In [ ]:
# Cell 8: Conditional CNN Model Implementation
from src.model import build_inference_model, get_model_config_summary

model = build_inference_model(config)
model.summary()

summary = get_model_config_summary(config)
print(f"\nEstimated parameters: {summary['estimated_params']:,}")

In [ ]:
# Cell 9: Model Parameter Count Verification
from src.utils import count_parameters

param_count = count_parameters(model)
print(f"Actual parameter count: {param_count:,}")
print(f"Expected: ~3,277")
assert param_count == 3277, f"Parameter count mismatch: {param_count} != 3277"

In [ ]:
# Cell 10: Test Forward Pass
import numpy as np

# Create dummy input: [batch, 28, 28, 12]
test_input = np.random.randn(1, 28, 28, 12).astype(np.float32)
output = model.predict(test_input, verbose=0)
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {output.shape}")
print(f"Output range: [{output.min():.3f}, {output.max():.3f}]")

## ⚠️ TRAINING SECTION - Long Running

The following cells perform model training. This may take 10-30 minutes on CPU.
Only execute when ready.

In [ ]:
# Cell 11: Prepare Training Data
from src.dataset import MNISTDataset

train_ds, val_ds, test_ds = dataset.prepare_datasets(
    validation_split=config['training']['validation_split'],
    batch_size=config['training']['batch_size'],
    seed=seed
)

print(f"Train batches: {tf.data.experimental.cardinality(train_ds).numpy()}")
print(f"Val batches: {tf.data.experimental.cardinality(val_ds).numpy()}")

In [ ]:
# Cell 12: Build Training Model
from src.train import DiffusionTrainer

trainer = DiffusionTrainer(config)
trainer.build_model()

In [ ]:
# Cell 13: Train Model (MAIN TRAINING CELL - Long Running!)
# ⚠️ This will train for the configured number of epochs
# history = trainer.train()

# To resume from checkpoint:
# history = trainer.train(resume_from='../artifacts/checkpoints/last_model.keras')

print("Training cell ready. Uncomment to run.")

In [ ]:
# Cell 14: Load Trained Model (after training)
# model_path = "../artifacts/checkpoints/best_model.keras"
# if os.path.exists(model_path):
#     model.load_weights(model_path)
#     print(f"Loaded weights from {model_path}")
# else:
#     print("No trained weights found. Using random initialization.")
print("Load trained model after training.")

In [ ]:
# Cell 15: Training History Visualization
from src.evaluate import plot_training_history
import json

history_path = "../artifacts/logs/training_history.json"
if os.path.exists(history_path):
    with open(history_path, 'r') as f:
        history = json.load(f)
    plot_training_history(history, save_path="../artifacts/training_history.png")
else:
    print("No training history found. Run training first.")

## DDIM Sampling Implementation

In [ ]:
# Cell 16: DDIM Scheduler
from src.scheduler import DDIMScheduler

scheduler = DDIMScheduler(config, noise_schedule)

print(f"DDIM timesteps (8 steps): {scheduler.get_ddim_timesteps(8)}")
print(f"DDIM timesteps (16 steps): {scheduler.get_ddim_timesteps(16)}")
print(f"DDIM timesteps (32 steps): {scheduler.get_ddim_timesteps(32)}")

In [ ]:
# Cell 17: Single Sample Generation
# Generate a single digit
sample = scheduler.sample(model, class_label=7, num_steps=8, eta=0.0)
print(f"Sample shape: {sample.shape}")
print(f"Sample range: [{sample.min():.3f}, {sample.max():.3f}]")

In [ ]:
# Cell 18: Visualize Generated Sample
import matplotlib.pyplot as plt
from src.utils import denormalize_to_zero_one

plt.figure(figsize=(3, 3))
plt.imshow(denormalize_to_zero_one(sample[..., 0]), cmap='gray', vmin=0, vmax=1)
plt.title("Generated Digit 7")
plt.axis('off')
plt.show()

In [ ]:
# Cell 19: Generate Class Grid (All Digits)
from src.scheduler import generate_class_grid
from src.evaluate import ModelEvaluator

evaluator = ModelEvaluator(config, model, noise_schedule)

classes = list(range(10))
samples_per_class = 5
num_steps = 8

class_grid = evaluator.generate_class_grid(classes, samples_per_class, num_steps)
print(f"Class grid shape: {class_grid.shape}")

In [ ]:
# Cell 20: Visualize Class Grid
evaluator.visualize_samples(class_grid, 
                           title=f"Generated Digits ({num_steps} DDIM steps)",
                           save_path="../artifacts/class_grid_8steps.png")

In [ ]:
# Cell 21: Sampling Steps Comparison
steps_list = [8, 16, 32]

for class_id in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    evaluator.visualize_step_comparison(class_id, steps_list,
                                       save_path=f"../artifacts/step_comparison_class_{class_id}.png")

## Evaluation

In [ ]:
# Cell 22: Full Evaluation
results = evaluator.run_full_evaluation("../artifacts/evaluation")

import json
print(json.dumps(results, indent=2, default=str))

In [ ]:
# Cell 23: Reproducibility Test
reproducible = evaluator.test_reproducibility()
print(f"Reproducible with fixed seed: {reproducible}")

## Model Export and Quantization

In [ ]:
# Cell 24: Export Float32 TFLite
from src.export import export_tflite_float32, export_tflite_int8, create_representative_dataset

ensure_dir("../artifacts")
export_tflite_float32(model, "../artifacts/model_float32.tflite")

In [ ]:
# Cell 25: Export INT8 TFLite
rep_data = create_representative_dataset(config, noise_schedule, 
                                         config['quantization']['representative_samples'])

int8_result = export_tflite_int8(model, config, noise_schedule,
                                 "../artifacts/model_int8.tflite",
                                 rep_data)
print(f"INT8 model size: {int8_result['size_bytes']} bytes")

In [ ]:
# Cell 26: Verify Quantization
from src.export import verify_quantized_model

verification = verify_quantized_model(
    "../artifacts/model_float32.tflite",
    "../artifacts/model_int8.tflite",
    config, noise_schedule,
    num_test_samples=100
)

In [ ]:
# Cell 27: TFLite Micro Compatibility Check
from src.utils import verify_tflite_ops

float_ops = verify_tflite_ops("../artifacts/model_float32.tflite")
int8_ops = verify_tflite_ops("../artifacts/model_int8.tflite")

print(f"Float32 compatible: {float_ops['is_micro_compatible']}")
print(f"INT8 compatible: {int8_ops['is_micro_compatible']}")

if float_ops['unsupported_ops']:
    print(f"Float unsupported: {float_ops['unsupported_ops']}")
if int8_ops['unsupported_ops']:
    print(f"INT8 unsupported: {int8_ops['unsupported_ops']}")

In [ ]:
# Cell 28: Model Size Report
import os

float_size = os.path.getsize("../artifacts/model_float32.tflite")
int8_size = os.path.getsize("../artifacts/model_int8.tflite")

print(f"Float32 TFLite: {float_size:,} bytes ({float_size/1024:.1f} KB)")
print(f"INT8 TFLite: {int8_size:,} bytes ({int8_size/1024:.1f} KB)")
print(f"Compression ratio: {float_size/int8_size:.2f}x")

## ESP32 Deployment Artifacts

In [ ]:
# Cell 29: Generate C Header for ESP32
from src.utils import create_model_header

ensure_dir("../esp32/include")
create_model_header("../artifacts/model_int8.tflite",
                   "../esp32/include/model_data.h",
                   "g_tiny_diffusion_model")

In [ ]:
# Cell 30: Verify Header
with open("../esp32/include/model_data.h", 'r') as f:
    header_content = f.read()

# Check model size in header
import re
match = re.search(r'const unsigned int \w+_len = (\d+);', header_content)
if match:
    print(f"Header model size: {match.group(1)} bytes")

# Show first few lines
print("\nHeader preview:")
print("\n".join(header_content.split("\n")[:15]))

In [ ]:
# Cell 31: Test INT8 Inference with TFLite Interpreter
import numpy as np

interpreter = tf.lite.Interpreter(model_path="../artifacts/model_int8.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

input_scale, input_zp = input_details['quantization']
output_scale, output_zp = output_details['quantization']

print(f"Input: scale={input_scale:.6f}, zp={input_zp}")
print(f"Output: scale={output_scale:.6f}, zp={output_zp}")

# Test inference
test_input = np.random.randn(1, 28, 28, 12).astype(np.float32)
test_input_int8 = np.round(test_input / input_scale + input_zp).astype(np.int8)

interpreter.set_tensor(input_details['index'], test_input_int8)
interpreter.invoke()
output_int8 = interpreter.get_tensor(output_details['index'])
output_float = (output_int8.astype(np.float32) - output_zp) * output_scale

print(f"Output shape: {output_float.shape}")
print(f"Output range: [{output_float.min():.3f}, {output_float.max():.3f}]")

## Deployment Instructions

### ESP32 Firmware Setup

1. **Hardware**: ESP32 DevKit V1 + SSD1306 OLED (128×64, I2C)
   - OLED SDA → GPIO 21
   - OLED SCL → GPIO 22
   - OLED VCC → 3.3V
   - OLED GND → GND

2. **PlatformIO**: Open `esp32/` folder in VS Code with PlatformIO extension

3. **Build**: `pio run` (compiles firmware)

4. **Upload**: `pio run --target upload` (flashes to ESP32)

5. **Monitor**: `pio device monitor` (115200 baud)

### Serial Commands

- `G 7` - Generate digit 7
- `G 3` - Generate digit 3
- `R` - Generate random digit
- `INFO` - Show memory and model info
- `HELP` - Show commands

### Memory Requirements

- Model: ~10-15 KB (INT8)
- Tensor arena: ~60 KB
- Total RAM usage: ~80-100 KB (well within 520 KB)
- Flash usage: ~50-100 KB

### Performance (Expected)

- 8 steps: ~200-400 ms total generation
- 16 steps: ~400-800 ms
- 32 steps: ~800-1600 ms
- Per denoising step: ~25-50 ms

### Known Limitations

- 8-step DDIM may show artifacts; 16/32 steps recommended for quality
- No PSRAM limits tensor arena size
- ESP-NN acceleration only for compatible ops
- Deterministic sampling (eta=0) by default

## Next Steps

1. Train the model (Cell 13)
2. Run evaluation (Cell 22)
3. Export and verify (Cells 24-28)
4. Build and flash ESP32 firmware
5. Test on hardware